In [ ]:
import sys
import subprocess
import datetime
import os
import glob
import traceback
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pulp
from prophet import Prophet
import warnings

# Import for parallelization
from joblib import Parallel, delayed

warnings.filterwarnings("ignore")

# Global Matplotlib configuration for clean rendering
%matplotlib inline
plt.rcParams.update({
    "font.size": 13,
    "axes.titlesize": 15,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,
})

# 1 — TariffCalculator
class TariffCalculator:
    GST_RATE = 0.10
    MLF = 0.995
    DLF = 1.045
    ENV_MARKET_RATE_KWH = 0.0250
    MONTHLY_SUBSCRIPTION_EX_GST = 20.00
    DAILY_SUPPLY_EX_GST         = 1.09
    DAYS_IN_MONTH               = 30

    @staticmethod
    def _network_rate_kwh(hour: int) -> float:
        if 15 <= hour < 21:
            return 0.2360
        elif 10 <= hour < 15:
            return 0.0270
        else:
            return 0.0720

    @classmethod
    def rates(cls, smp_market_price_mwh: float, utc_date: datetime.datetime) -> tuple:
        spot_price_kwh    = smp_market_price_mwh / 0.615
        adjusted_spot_kwh  = spot_price_kwh * cls.MLF * cls.DLF
        nem_time = utc_date + datetime.timedelta(hours=10)
        network_rate_kwh = cls._network_rate_kwh(nem_time.hour)
        buy_rate  = (adjusted_spot_kwh + network_rate_kwh + cls.ENV_MARKET_RATE_KWH) * (1 + cls.GST_RATE)
        sell_rate = adjusted_spot_kwh
        return buy_rate, sell_rate

    @classmethod
    def rates_series(cls, smp_series: pd.Series) -> tuple:
        idx = smp_series.index
        buy_arr  = np.empty(len(smp_series))
        sell_arr = np.empty(len(smp_series))
        for i, (ts, smp) in enumerate(zip(idx, smp_series.values)):
            py_ts = ts.to_pydatetime() if hasattr(ts, "to_pydatetime") else ts
            b, s = cls.rates(float(smp), py_ts)
            buy_arr[i]  = b
            sell_arr[i] = s
        return buy_arr, sell_arr

    @classmethod
    def constant_cost_per_interval(cls, interval_minutes: int = 30) -> float:
        intervals_per_day   = (24 * 60) / interval_minutes
        intervals_per_month = intervals_per_day * cls.DAYS_IN_MONTH
        constant_cost_ex_gst = (
            cls.DAILY_SUPPLY_EX_GST / intervals_per_day
            + cls.MONTHLY_SUBSCRIPTION_EX_GST / intervals_per_month
        )
        return constant_cost_ex_gst * (1 + cls.GST_RATE)


# 2 — EnergyForecaster
class EnergyForecaster:
    def __init__(self):
        self.model_con = Prophet(
            seasonality_mode="additive",
            daily_seasonality=True,
            weekly_seasonality=True,
            yearly_seasonality=False,
            changepoint_prior_scale=0.05,
            uncertainty_samples=0, 
        )
        self.model_gen = Prophet(
            seasonality_mode="multiplicative",
            daily_seasonality=True,
            weekly_seasonality=False,
            yearly_seasonality=False,
            changepoint_prior_scale=0.05,
            uncertainty_samples=0,
        )
        self._fitted = False

    @staticmethod
    def _to_prophet_df(series: pd.Series) -> pd.DataFrame:
        df = series.reset_index()
        df.columns = ["ds", "y"]
        df["ds"] = pd.to_datetime(df["ds"]).dt.tz_localize(None)
        df["y"]  = df["y"].clip(lower=0)
        return df

    def fit(self, df: pd.DataFrame, col_con: str, col_gen: str) -> None:
        self.model_con.fit(self._to_prophet_df(df[col_con]))
        self.model_gen.fit(self._to_prophet_df(df[col_gen]))
        self._fitted = True

    def predict_next_day(self, anchor_ts: pd.Timestamp, horizon_steps: int = 48, freq: str = "30min") -> pd.DataFrame:
        if not self._fitted:
            raise RuntimeError("Call .fit() before predicting.")
        ts = anchor_ts.tz_localize(None) if anchor_ts.tzinfo else anchor_ts
        future = pd.DataFrame({"ds": pd.date_range(ts, periods=horizon_steps, freq=freq)})
        
        fc_con = self.model_con.predict(future)[["ds", "yhat"]].rename(columns={"yhat": "yhat_con"})
        fc_con["yhat_con"] = fc_con["yhat_con"].clip(lower=0)
        
        fc_gen = self.model_gen.predict(future)[["ds", "yhat"]].rename(columns={"yhat": "yhat_gen"})
        fc_gen["yhat_gen"] = fc_gen["yhat_gen"].clip(lower=0)
        
        return fc_con.merge(fc_gen, on="ds")


# 3 — MILPScheduler
class MILPScheduler:
    def __init__(self, battery_cap: float = 20.0, soc_min_pct: float = 0.10, soc_max_pct: float = 0.80, p_max: float = 1.5, eff: float = 0.95, delta_t: float = 0.5):
        self.battery_cap = battery_cap
        self.soc_min  = battery_cap * soc_min_pct
        self.soc_max  = battery_cap * soc_max_pct
        self.p_max    = p_max
        self.eff      = eff
        self.delta_t  = delta_t

    def solve(self, soc_init: float, buy_rate: list, sell_rate: list, p_gen: list, p_con: list, terminal_soc: float | None = None) -> dict:
        H    = len(buy_rate)
        soc0 = terminal_soc if terminal_soc is not None else soc_init
        mdl = pulp.LpProblem("MILP_HEMS", pulp.LpMinimize)

        x_ch   = pulp.LpVariable.dicts("ch",   range(H), lowBound=0, upBound=self.p_max)
        x_dis  = pulp.LpVariable.dicts("dis",  range(H), lowBound=0, upBound=self.p_max)
        p_buy  = pulp.LpVariable.dicts("buy",  range(H), lowBound=0)
        p_sell = pulp.LpVariable.dicts("sell", range(H), lowBound=0)
        SoC    = pulp.LpVariable.dicts("soc",  range(H), lowBound=self.soc_min, upBound=self.soc_max)
        d_ch   = pulp.LpVariable.dicts("dch",  range(H), cat="Binary")
        d_dis  = pulp.LpVariable.dicts("ddis", range(H), cat="Binary")

        mdl += pulp.lpSum(p_buy[t] * buy_rate[t] * self.delta_t - p_sell[t] * sell_rate[t] * self.delta_t for t in range(H)), "MinNetCost"

        for t in range(H):
            soc_prev = soc_init if t == 0 else SoC[t - 1]
            mdl += (p_con[t] + x_ch[t] + p_sell[t] == p_gen[t] + x_dis[t] + p_buy[t]), f"balance_{t}"
            mdl += d_ch[t] + d_dis[t] <= 1, f"mutex_{t}"
            mdl += x_ch[t]  <= self.p_max * d_ch[t],  f"ch_bound_{t}"
            mdl += x_dis[t] <= self.p_max * d_dis[t], f"dis_bound_{t}"
            mdl += (SoC[t] == soc_prev + (x_ch[t] * self.eff - x_dis[t] / self.eff) * self.delta_t), f"soc_dyn_{t}"

        mdl += SoC[H - 1] >= soc0, "terminal"
        mdl.solve(pulp.PULP_CBC_CMD(msg=0))
        status = pulp.LpStatus[mdl.status]

        if status != "Optimal":
            return {
                "status": status, "x_ch": [0.0] * H, "x_dis": [0.0] * H,
                "p_buy": [0.0] * H, "p_sell": [0.0] * H, "soc_plan": [soc_init] * H, "cost": 0.0,
            }

        return {
            "status": status,
            "x_ch": [pulp.value(x_ch[t]) or 0.0 for t in range(H)],
            "x_dis": [pulp.value(x_dis[t]) or 0.0 for t in range(H)],
            "p_buy": [pulp.value(p_buy[t]) or 0.0 for t in range(H)],
            "p_sell": [pulp.value(p_sell[t]) or 0.0 for t in range(H)],
            "soc_plan": [pulp.value(SoC[t]) or 0.0 for t in range(H)],
            "cost": pulp.value(mdl.objective) or 0.0,
        }


# 4 — ReactiveController
class ReactiveController:
    def __init__(self, scheduler: MILPScheduler, forecaster: EnergyForecaster, real_data: pd.DataFrame, soc_init: float = 10.0, horizon_steps: int = 48, soc_deviation_threshold: float = 0.5, reoptimize_every: int = 1, freq: str = "30min"):
        self.sched    = scheduler
        self.fc       = forecaster
        self.data     = real_data
        self.soc0     = soc_init
        self.H        = horizon_steps
        self.dev_thr  = soc_deviation_threshold
        self.reopt_n  = reoptimize_every
        self.freq     = freq
        self._fc_cache: dict = {}
        buy_arr, sell_arr = TariffCalculator.rates_series(self.data["SMP"])
        self.buy_rate  = buy_arr
        self.sell_rate = sell_arr

    def _real_slice(self, k: int, h: int) -> tuple:
        sl = self.data.iloc[k : k + h]
        return (self.buy_rate[k : k + h].tolist(), self.sell_rate[k : k + h].tolist(), sl["Energy_Generation"].tolist(), sl["Energy_Consumption"].tolist())

    def _forecast_slice(self, k: int, h: int) -> tuple:
        day_idx = k // self.H
        day_offset = k % self.H
        for d in [day_idx, day_idx + 1]:
            if d not in self._fc_cache:
                start_k = d * self.H
                if start_k < len(self.data):
                    anchor = self.data.index[start_k]
                    self._fc_cache[d] = self.fc.predict_next_day(anchor, self.H, freq=self.freq)

        fc_today = self._fc_cache[day_idx]
        fc_sl    = fc_today.iloc[day_offset : day_offset + h].reset_index(drop=True)

        if len(fc_sl) < h and (day_idx + 1) in self._fc_cache:
            missing     = h - len(fc_sl)
            fc_tomorrow = self._fc_cache[day_idx + 1]
            fc_next     = fc_tomorrow.iloc[:missing].reset_index(drop=True)
            fc_sl       = pd.concat([fc_sl, fc_next], ignore_index=True)

        if len(fc_sl) < h:
            pad  = h - len(fc_sl)
            last = fc_sl.iloc[[-1]]
            fc_sl = pd.concat([fc_sl] + [last] * pad, ignore_index=True)

        buy_rate_real  = self.buy_rate[k : k + h].tolist()
        sell_rate_real = self.sell_rate[k : k + h].tolist()
        p_gen_fc = fc_sl["yhat_gen"].tolist()
        p_con_fc = fc_sl["yhat_con"].tolist()
        p_gen_fc[0] = self.data["Energy_Generation"].iloc[k]
        p_con_fc[0] = self.data["Energy_Consumption"].iloc[k]
        return buy_rate_real, sell_rate_real, p_gen_fc, p_con_fc

    def run(self, num_days: int = 5, use_forecast: bool = True) -> pd.DataFrame:
        T_total  = min(num_days * self.H, len(self.data))
        soc_cur  = self.soc0
        plan     = None
        plan_pos = 0
        history  = []

        for k in range(T_total):
            horizon = min(self.H, T_total - k, len(self.data) - k)
            if horizon <= 0: break
            soc_dev = 0.0
            if plan is not None and plan_pos < len(plan["soc_plan"]):
                soc_dev = abs(soc_cur - plan["soc_plan"][plan_pos])

            need_reopt = (plan is None or plan_pos >= len(plan["x_ch"]) or k % self.reopt_n == 0 or soc_dev > self.dev_thr)
            if need_reopt:
                fn = self._forecast_slice if use_forecast else self._real_slice
                buy_h, sell_h, p_gen_h, p_con_h = fn(k, horizon)
                plan     = self.sched.solve(soc_cur, buy_h, sell_h, p_gen_h, p_con_h)
                plan_pos = 0

            act_ch   = plan["x_ch"][plan_pos]
            act_dis  = plan["x_dis"][plan_pos]
            real_smp       = self.data["SMP"].iloc[k]
            real_buy_rate  = self.buy_rate[k]
            real_sell_rate = self.sell_rate[k]
            real_gen       = self.data["Energy_Generation"].iloc[k]
            real_con       = self.data["Energy_Consumption"].iloc[k]

            p_net_real = real_con + act_ch - real_gen - act_dis
            if p_net_real > 0:
                act_buy  = p_net_real
                act_sell = 0.0
            else:
                act_buy  = 0.0
                act_sell = -p_net_real

            delta_soc = (act_ch * self.sched.eff - act_dis / self.sched.eff) * self.sched.delta_t
            soc_cur   = float(np.clip(soc_cur + delta_soc, self.sched.soc_min, self.sched.soc_max))
            step_cost = (act_buy * real_buy_rate - act_sell * real_sell_rate) * self.sched.delta_t

            history.append({
                "Timestamp":        self.data.index[k],
                "Price_SMP":        real_smp,
                "Buy_Rate_USD_kWh": real_buy_rate,
                "Sell_Rate_USD_kWh":real_sell_rate,
                "Solar_Gen":        real_gen,
                "Consumption":      real_con,
                "SoC_kWh":          soc_cur,
                "SoC_Planned":      plan["soc_plan"][plan_pos],
                "SoC_Deviation":    soc_dev,
                "Charge_kW":        act_ch,
                "Discharge_kW":     act_dis,
                "Buy_kW":           act_buy,
                "Sell_kW":          act_sell,
                "Step_Cost_USD":    step_cost,
                "Reoptimized":      int(need_reopt),
            })
            plan_pos += 1
        return pd.DataFrame(history).set_index("Timestamp")


# 5 — KPITracker
class KPITracker:
    @staticmethod
    def compare_three(df_fc: pd.DataFrame, df_pk: pd.DataFrame, delta_t: float = 0.5) -> pd.DataFrame:
        buy_nb  = np.maximum(0, df_fc["Consumption"] - df_fc["Solar_Gen"])
        sell_nb = np.maximum(0, df_fc["Solar_Gen"]   - df_fc["Consumption"])
        cost_nb = ((buy_nb * df_fc["Buy_Rate_USD_kWh"] - sell_nb * df_fc["Sell_Rate_USD_kWh"]) * delta_t).sum()
        cost_pk = df_pk["Step_Cost_USD"].sum()
        cost_fc = df_fc["Step_Cost_USD"].sum()

        fixed_cost = TariffCalculator.constant_cost_per_interval(interval_minutes=int(delta_t * 60)) * len(df_fc)
        cost_nb += fixed_cost
        cost_pk += fixed_cost
        cost_fc += fixed_cost

        buy_nb_e = (buy_nb * delta_t).sum()
        buy_pk   = (df_pk["Buy_kW"] * delta_t).sum()
        buy_fc   = (df_fc["Buy_kW"] * delta_t).sum()
        sell_nb_e = (sell_nb * delta_t).sum()
        sell_pk   = (df_pk["Sell_kW"] * delta_t).sum()
        sell_fc   = (df_fc["Sell_kW"] * delta_t).sum()

        rows = [
            {"KPI": "Total cost (USD)", "No battery": round(cost_nb, 2), "Perfect foresight": round(cost_pk, 2), "Prophet forecast": round(cost_fc, 2)},
            {"KPI": "Savings vs no battery (USD)", "No battery": 0.0, "Perfect foresight": round(cost_nb - cost_pk, 2), "Prophet forecast": round(cost_nb - cost_fc, 2)},
            {"KPI": "Regret vs perfect foresight (USD)", "No battery": "—", "Perfect foresight": 0.0, "Prophet forecast": round(cost_fc - cost_pk, 2)},
            {"KPI": "Energy purchased (kWh)", "No battery": round(buy_nb_e, 1), "Perfect foresight": round(buy_pk, 1), "Prophet forecast": round(buy_fc, 1)},
            {"KPI": "Energy sold (kWh)", "No battery": round(sell_nb_e, 1), "Perfect foresight": round(sell_pk, 1), "Prophet forecast": round(sell_fc, 1)},
        ]
        return pd.DataFrame(rows).set_index("KPI")

    @staticmethod
    def extract_kpi_raw(kpi_table: pd.DataFrame) -> dict:
        return {
            "cost_no_battery": float(kpi_table.loc["Total cost (USD)", "No battery"]),
            "cost_oracle":     float(kpi_table.loc["Total cost (USD)", "Perfect foresight"]),
            "cost_prophet":    float(kpi_table.loc["Total cost (USD)", "Prophet forecast"]),
        }



# 6 — HEMSVisualizer (Optimized for parallel execution)
class HEMSVisualizer:
    @staticmethod
    def plot_scenarios(df_fc, df_pk, delta_t=0.5, save_path=None, title_suffix=""):
        buy_nb  = np.maximum(0, df_fc["Consumption"] - df_fc["Solar_Gen"])
        sell_nb = np.maximum(0, df_fc["Solar_Gen"]   - df_fc["Consumption"])
        fixed = TariffCalculator.constant_cost_per_interval(interval_minutes=int(delta_t * 60))

        cum_nb = (((buy_nb * df_fc["Buy_Rate_USD_kWh"] - sell_nb * df_fc["Sell_Rate_USD_kWh"]) * delta_t) + fixed).cumsum()
        cum_fc = (df_fc["Step_Cost_USD"] + fixed).cumsum()
        cum_pk = (df_pk["Step_Cost_USD"] + fixed).cumsum()

        gain_pk = cum_nb - cum_pk
        gain_fc = cum_nb - cum_fc
        regret  = cum_fc - cum_pk

        fig, ax = plt.subplots(figsize=(15, 7))
        ax.fill_between(df_fc.index, cum_pk, cum_fc, alpha=0.25, color="orange", label="Regret")
        ax.fill_between(df_fc.index, cum_fc, cum_nb, alpha=0.15, color="steelblue", label="Prophet gain")
        ax.plot(df_fc.index, cum_nb, color="tomato", lw=2, label="No battery")
        ax.plot(df_pk.index, cum_pk, color="darkgreen", lw=2, ls="--", label="Perfect foresight")
        ax.plot(df_fc.index, cum_fc, color="steelblue", lw=2, label="Prophet forecast")

        summary = (
            f"Summary\n"
            f"Oracle gain: {gain_pk.iloc[-1]:.2f} USD\n"
            f"Prophet gain: {gain_fc.iloc[-1]:.2f} USD\n"
            f"Regret: {regret.iloc[-1]:.2f} USD"
        )
        ax.text(
            0.02, 0.98, summary, transform=ax.transAxes, va="top",
            fontsize=16,
            bbox=dict(boxstyle="round,pad=0.7", facecolor="white",
                      edgecolor="black", linewidth=0.8, alpha=0.95),
        )

        ax.set_ylabel("Cumulative cost (USD)")
        ax.set_title(f"Scenario comparison {title_suffix}".strip())
        ax.grid(alpha=0.3)
        ax.legend(fontsize=15, loc="lower right")

        n_days = (df_fc.index[-1] - df_fc.index[0]).days + 1
        if n_days <= 3: locator, fmt = mdates.HourLocator(interval=6), "%d/%m %Hh"
        elif n_days <= 14: locator, fmt = mdates.DayLocator(interval=1), "%d/%m"
        elif n_days <= 60: locator, fmt = mdates.DayLocator(interval=5), "%d/%m"
        else: locator, fmt = mdates.MonthLocator(interval=1), "%m/%Y"

        ax.xaxis.set_major_locator(locator)
        ax.xaxis.set_major_formatter(mdates.DateFormatter(fmt))
        plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
        plt.tight_layout()
        if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.close(fig)  

    @staticmethod
    def save_table_as_image(df, save_path="kpi_results.png"):
        fig, ax = plt.subplots(figsize=(12, 3))
        ax.axis("off")
        table = ax.table(cellText=df.values, rowLabels=df.index, colLabels=df.columns, cellLoc="center", loc="center")
        table.auto_set_font_size(False)
        table.set_fontsize(13)
        table.scale(1.2, 2.0)
        for (row, col), cell in table.get_celld().items():
            if row == 0:
                cell.set_facecolor("#40466e")
                cell.set_text_props(color="white", weight="bold")
            if col == -1:
                cell.set_facecolor("#f2f2f2")
                cell.set_text_props(weight="bold")
        plt.tight_layout()
        if save_path: plt.savefig(save_path, dpi=300, bbox_inches="tight")
        plt.close(fig)

    @staticmethod
    def plot_battery_soc(df_fc, df_pk=None, save_path=None, title_suffix=""):
        fig, ax = plt.subplots(figsize=(15, 5))
        ax.plot(df_fc.index, df_fc["SoC_kWh"], label="SoC - Prophet forecast", color="steelblue", linewidth=2)
        if df_pk is not None:
            ax.plot(df_pk.index, df_pk["SoC_kWh"], "--", label="SoC - Perfect foresight", color="darkgreen", linewidth=2)
        ax.set_ylabel("Stored energy (kWh)")
        ax.set_xlabel("Time")
        ax.set_title(f"Battery state of charge (SoC) {title_suffix}".strip())
        ax.grid(alpha=0.3)
        ax.legend()
        plt.tight_layout()
        if save_path: plt.savefig(save_path, dpi=200, bbox_inches="tight")
        plt.close(fig)

    @staticmethod
    def plot_exact_match_boxplot(values, ylabel, save_path=None, group_label="MILP+Prophet"):
        values = np.asarray(values, dtype=float)
        values = values[~np.isnan(values)]
        
        fig, ax = plt.subplots(figsize=(3.8, 5.0))
        
        ax.boxplot(values, positions=[1], widths=0.55, showfliers=True,
                   flierprops=dict(marker="o", markerfacecolor="none", markeredgecolor="black", markersize=7, markeredgewidth=1.3),
                   medianprops=dict(color="#ff7f0e", linewidth=1.8),
                   boxprops=dict(color="black", linewidth=1),
                   whiskerprops=dict(color="black", linewidth=1),
                   capprops=dict(color="black", linewidth=1))
        
        rng = np.random.default_rng(42)  
        jitter_x = rng.normal(loc=1.0, scale=0.032, size=len(values))
        ax.scatter(jitter_x, values, alpha=0.45, color="#347bb7", edgecolor="none", s=28, zorder=3)
        
        ax.set_xlim(0.55, 1.45)
        ax.set_xticks([1])
        ax.set_xticklabels([group_label])
        ax.set_ylabel(ylabel)
        
        ax.grid(True, which="both", color="#e8e8e8", linestyle="-", linewidth=1.0, zorder=1)
        ax.set_axisbelow(True)
        
        plt.tight_layout()
        if save_path: 
            plt.savefig(save_path, dpi=250, bbox_inches="tight")
        plt.show()  # Only the final boxplots are displayed on screen


# 7 — Pipeline for 1 dataset
def run_pipeline_for_file(file_path: str, output_root: str = "results", battery_cap: float = 20.0, soc_min_pct: float = 0.10, soc_max_pct: float = 0.80, p_max: float = 1.5, eff: float = 0.95, delta_t: float = 0.5, soc_init: float = 10.0, H: int = 48, n_train: int = 730, n_sim: int = 365, start_ts: str = "2010-07-01 00:30:00") -> dict:
    
    dataset_name = os.path.splitext(os.path.basename(file_path))[0]
    out_dir = os.path.join(output_root, dataset_name)
    os.makedirs(out_dir, exist_ok=True)

    print(f"👉 [Starting] Processing: {dataset_name}")

    try:
        raw = pd.read_csv(file_path)
        raw.index = pd.to_datetime(raw["Timestamp_UTC"], format="ISO8601")
        df_all = raw.loc[start_ts:, ["SMP", "Energy_Generation", "Energy_Consumption"]].copy()

        df_train = df_all.iloc[: n_train * H]
        df_sim   = df_all.iloc[n_train * H : (n_train + n_sim) * H]

        if len(df_train) == 0 or len(df_sim) == 0:
            print(f"[Error] Insufficient data for {dataset_name}")
            return None

        forecaster = EnergyForecaster()
        forecaster.fit(df_train, "Energy_Consumption", "Energy_Generation")

        scheduler = MILPScheduler(battery_cap=battery_cap, soc_min_pct=soc_min_pct, soc_max_pct=soc_max_pct, p_max=p_max, eff=eff, delta_t=delta_t)
        controller = ReactiveController(scheduler=scheduler, forecaster=forecaster, real_data=df_sim, soc_init=soc_init, horizon_steps=H, freq="30min")
        
        df_fc = controller.run(num_days=n_sim, use_forecast=True)
        df_pk = controller.run(num_days=n_sim, use_forecast=False)

        kpi_table = KPITracker.compare_three(df_fc, df_pk, delta_t=delta_t)
        kpi_raw = KPITracker.extract_kpi_raw(kpi_table)
        kpi_raw["dataset"] = dataset_name

        # Silent save to disk (no plt.show() here, for performance)
        HEMSVisualizer.plot_scenarios(df_fc, df_pk, delta_t=delta_t, save_path=os.path.join(out_dir, "scenarios.png"), title_suffix=f"- {dataset_name}")
        HEMSVisualizer.plot_battery_soc(df_fc, df_pk, save_path=os.path.join(out_dir, "battery_soc.png"), title_suffix=f"- {dataset_name}")
        HEMSVisualizer.save_table_as_image(kpi_table, save_path=os.path.join(out_dir, "kpi_table.png"))

        print(f"✅ [Done] File processed successfully: {dataset_name}")
        return kpi_raw

    except Exception as e:
        print(f"❌ [Error] Crash on file {dataset_name}: {e}")
        return None


# 8 — MAIN
def main():
    BATTERY_CAP = 10.0
    SOC_MIN_PCT = 0.10
    SOC_MAX_PCT = 0.80
    P_MAX       = 1.5
    EFF         = 0.95
    DELTA_T     = 0.5    
    SOC_INIT    = 5.0
    H           = 48     
    N_TRAIN     = 730
    N_SIM       = 365
    
    DATA_DIR = "./data"          
    OUTPUT_ROOT = "./results"    

    file_paths = glob.glob(os.path.join(DATA_DIR, "*.csv"))
    if not file_paths:
        print(f"[Error] No CSV file found in '{DATA_DIR}'.")
        return

    print(f"=== Initializing PARALLEL HEMS Pipeline ===")
    print(f"Files detected: {len(file_paths)}")
    print(f"Using ALL available processor cores...")


    results = Parallel(n_jobs=-1)(
        delayed(run_pipeline_for_file)(
            file_path=fp, 
            output_root=OUTPUT_ROOT, 
            battery_cap=BATTERY_CAP,
            soc_min_pct=SOC_MIN_PCT,
            soc_max_pct=SOC_MAX_PCT,
            p_max=P_MAX,
            eff=EFF,
            delta_t=DELTA_T,
            soc_init=SOC_INIT,
            H=H,
            n_train=N_TRAIN,
            n_sim=N_SIM
        ) for fp in file_paths
    )

    all_kpis = [res for res in results if res is not None]

    if all_kpis:
        print("\n=== Final Step: Computing Metrics and Boxplots ===")
        df_global = pd.DataFrame(all_kpis)
        
        # 1. Cost improvement vs no battery (%) 
        cost_improvement_pct = ((df_global["cost_no_battery"] - df_global["cost_prophet"]) / df_global["cost_no_battery"]) * 100
        
        # 2. Fraction of the oracle's theoretical gain captured (%) 
        gain_oracle = df_global["cost_no_battery"] - df_global["cost_oracle"]
        gain_prophet = df_global["cost_no_battery"] - df_global["cost_prophet"]
        fraction_theoretical_gain_pct = (gain_prophet / np.where(gain_oracle == 0, 1e-6, gain_oracle)) * 100

        median_improvement = np.median(cost_improvement_pct.dropna())
        median_fraction = np.median(fraction_theoretical_gain_pct.dropna())
        
        print("\n" + "="*70)
        print(" GLOBAL STATISTICAL RESULTS (ORANGE LINE ON THE CHARTS):")
        print(f" Chart 1 (image_8dcacf.png) -> Median value: {median_improvement:.2f} %")
        print(f" Chart 2 (image_8dcaee.png) -> Median value: {median_fraction:.2f} %")
        print("="*70 + "\n")

        print("Displaying Boxplot: Cost Improvement (%)")
        path_box1 = os.path.join(OUTPUT_ROOT, "global_cost_improvement_boxplot.png")
        HEMSVisualizer.plot_exact_match_boxplot(
            values=cost_improvement_pct, 
            ylabel="Cost improvement vs. no battery (%)", 
            save_path=path_box1
        )
        
        print("\nDisplaying Boxplot: Fraction of Theoretical Gain (%)")
        path_box2 = os.path.join(OUTPUT_ROOT, "global_fraction_gain_boxplot.png")
        HEMSVisualizer.plot_exact_match_boxplot(
            values=fraction_theoretical_gain_pct, 
            ylabel="Fraction of theoretical gain (%)", 
            save_path=path_box2
        )
        
        print(f"\nProcessing complete! Final charts have been saved in '{OUTPUT_ROOT}'.")

# Launch code
if __name__ == "__main__":
    main()

c:\Users\Yoann\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Importing plotly failed. Interactive plots will not work.


=== Initialisation du Pipeline HEMS — Ausgrid 65 uniquement ===
👉 [Lancement] Traitement de : Ausgrid 65


08:29:24 - cmdstanpy - INFO - Chain [1] start processing
08:29:32 - cmdstanpy - INFO - Chain [1] done processing
08:29:33 - cmdstanpy - INFO - Chain [1] start processing
08:29:41 - cmdstanpy - INFO - Chain [1] done processing


✅ [Terminé] Fichier traité avec succès : Ausgrid 65

=== Résultat Ausgrid 65 ===
cost_no_battery: 1174.94
cost_oracle: 984.7
cost_prophet: 997.2
dataset: Ausgrid 65

Graphiques sauvegardés dans './results/Ausgrid 65/'.
